In [6]:
import os
import json
import numpy
import datetime
import certifi
import pandas as pd

import pymongo
import sqlalchemy
from sqlalchemy import create_engine, text

In [7]:
print(f"Running SQL Alchemy Version: {sqlalchemy.__version__}")
print(f"Running PyMongo Version: {pymongo.__version__}")

Running SQL Alchemy Version: 2.0.34
Running PyMongo Version: 4.15.3


#### Declare & Assign Connection Variables for the MySQL Server & Databases

In [8]:
host_name = "localhost"
port = "3306"
uid = "root"
pwd = "Point03%"

src_dbname = "chinook"
dst_dbname = "chinook_dw"

# The 'cluster_location' must either be "atlas" or "local".
mongodb_args = {
    "user_name" : "cth9ss",
    "password" : "929395",
    "cluster_name" : "ds2002",
    "cluster_subnet" : "u1yniz4",
    "cluster_location" : "atlas", # "local"
    "db_name" : "chinook_customer"
}

#### Define Functions for Getting Data From and Setting Data Into Databases

In [21]:
def get_dataframe(uid, pwd, host_name, db_name, sql_query):
    conn_str = f"mysql+pymysql://{uid}:{pwd}@{host_name}/{db_name}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    dframe = pd.read_sql(sql_query, connection);
    connection.close()
    
    return dframe


def set_dataframe(uid, pwd, host_name, db_name, df, table_name, pk_column, db_operation):
    conn_str = f"mysql+pymysql://{uid}:{pwd}@{host_name}/{db_name}"
    sqlEngine = create_engine(conn_str, pool_recycle=3600)
    connection = sqlEngine.connect()
    
    if db_operation == "insert":
        df.to_sql(table_name, con=connection, index=False, if_exists='replace')
        connection.execute(text(f"ALTER TABLE {table_name} ADD PRIMARY KEY ({pk_column});"))
            
    elif db_operation == "update":
        df.to_sql(table_name, con=connection, index=False, if_exists='append')
    
    connection.close()


def get_mongo_client(**args):
    '''Validate proper input'''
    if args["cluster_location"] not in ['atlas', 'local']:
        raise Exception("You must specify either 'atlas' or 'local' for the cluster_location parameter.")
    
    else:
        if args["cluster_location"] == "atlas":
            connect_str = f"mongodb+srv://{args['user_name']}:{args['password']}@"
            connect_str += f"{args['cluster_name']}.{args['cluster_subnet']}.mongodb.net"
            client = pymongo.MongoClient(connect_str, tlsCAFile=certifi.where())
            
        elif args["cluster_location"] == "local":
            client = pymongo.MongoClient("mongodb://localhost:27017/")
        
    return client


def get_mongo_dataframe(mongo_client, db_name, collection, query):
    '''Query MongoDB, and fill a python list with documents to create a DataFrame'''
    db = mongo_client[db_name]
    dframe = pd.DataFrame(list(db[collection].find(query)))
    dframe.drop(['_id'], axis=1, inplace=True)
    mongo_client.close()
    
    return dframe


def set_mongo_collections(mongo_client, db_name, data_directory, json_files):
    db = mongo_client[db_name]
    
    for file in json_files:
        db.drop_collection(file)
        json_file = os.path.join(data_directory, json_files[file])
        with open(json_file, 'r', encoding='utf-8') as openfile:  # <-- explicit UTF-8
            json_object = json.load(openfile)
            collection = db[file]
            result = collection.insert_many(json_object)
        
    mongo_client.close()

        
    mongo_client.close()

#### Create the New Data Warehouse database, and to Use it, Switch the Connection Context.
*Drop* database if it already exists, then *create* the new **chinook_dw** database and *use* it as the target of all subsequent operations.

In [62]:
conn_str = f"mysql+pymysql://{uid}:{pwd}@{host_name}"
sqlEngine = create_engine(conn_str, pool_recycle=3600)
connection = sqlEngine.connect()

connection.execute(text(f"DROP DATABASE IF EXISTS `{dst_dbname}`;"))
connection.execute(text(f"CREATE DATABASE `{dst_dbname}`;"))
connection.execute(text(f"USE {dst_dbname};"))

connection.close()

### Create & Populate the Dimension Tables

#### Extract Data from the Source Database Tables
Fetch data for each dimension table (track, album, artist, genre) from the **chinook** database using the **get_dataframe()** function.

In [63]:
sql_track = "SELECT * FROM chinook.track;"
df_track = get_dataframe(uid, pwd, host_name, src_dbname, sql_track)
df_track.head(2)

,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
0,1,For Those About To Rock (We Salute You),1,1,1,"Angus Young, Malcolm Young, Brian Johnson",343719,11170334,0.99
1,2,Balls to the Wall,2,2,1,"U. Dirkschneider, W. Hoffmann, H. Frank, P. Ba...",342562,5510424,0.99


In [64]:
sql_album = "SELECT * FROM chinook.album;"
df_album = get_dataframe(uid, pwd, host_name, src_dbname, sql_album)
df_album.head(2)

,AlbumId,Title,ArtistId
0,1,For Those About To Rock We Salute You,1
1,2,Balls to the Wall,2


In [65]:
sql_artist = "SELECT * FROM chinook.artist;"
df_artist = get_dataframe(uid, pwd, host_name, src_dbname, sql_artist)
df_artist.head(2)

,ArtistId,Name
0,1,AC/DC
1,2,Accept


In [66]:
sql_genre = "SELECT * FROM chinook.genre;"
df_genre = get_dataframe(uid, pwd, host_name, src_dbname, sql_genre)
df_genre.head(2)

,GenreId,Name
0,1,Rock
1,2,Jazz


In [67]:
# EXTRACT DATA FROM .CSV FILE
data_dir = os.path.join(os.getcwd(), 'data') 
mediatype_csv = os.path.join(data_dir, 'chinook_mediatype.csv')

df_mediatype = pd.read_csv(mediatype_csv, encoding='utf-8')
df_mediatype.head(2)

,MediaTypeId,Name
0,1,MPEG audio file
1,2,Protected AAC audio file


In [68]:
# POPULATE MONGODB W/ SOURCE DATA
client = get_mongo_client(**mongodb_args)

# Gets the path of the Current Working Directory for this Notebook,
# and then Appends the 'data' directory.
data_dir = os.path.join(os.getcwd(), 'data')

json_files = {"customers" : 'chinook_customer.json'}

set_mongo_collections(client, mongodb_args["db_name"], data_dir, json_files)         

In [69]:
# TODO: Extract data from the "customers" collection

client = get_mongo_client(**mongodb_args)

query = {}
collection = "customers"

df_customer = get_mongo_dataframe(client, mongodb_args["db_name"], collection, query)
df_customer.head(2)


,CustomerId,FirstName,LastName,Company,Address,City,State,Country,PostalCode,Phone,Fax,Email,SupportRepId
0,1,Luís,Gonçalves,Embraer - Empresa Brasileira de Aeronáutica S.A.,"Av. Brigadeiro Faria Lima, 2170",São José dos Campos,SP,Brazil,12227-000,+55 (12) 3923-5555,+55 (12) 3923-5566,luisg@embraer.com.br,3
1,2,Leonie,Köhler,None,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,+49 0711 2842222,None,leonekohler@surfeu.de,5


#### Create the Date Dimension Table
At this point, **execute the populate_dim_date SQL script** that creates and populates a **Date Dimension** table.

#### Perform Any Necessary Transformations

In [60]:
print("track columns:", df_track.columns.tolist())
print("album columns:", df_album.columns.tolist())
print("artist columns:", df_artist.columns.tolist())
print("genre columns:", df_genre.columns.tolist())
print("mediatype columns:", df_mediatype.columns.tolist())


track columns: ['TrackId', 'Name', 'AlbumId', 'MediaTypeId', 'GenreId', 'Composer', 'Milliseconds', 'Bytes', 'UnitPrice']
album columns: ['AlbumId', 'Title', 'ArtistId']
artist columns: ['ArtistId', 'Name']
genre columns: ['GenreId', 'Name']
mediatype columns: ['MediaTypeId', 'Name']


In [71]:
df_mediatype.rename(columns={'Name': 'MediaTypeName'}, inplace=True)

# merge track with album, artist, and genre
df_track = (
    df_track
    .merge(df_album, on='AlbumId', how='left')
    .merge(df_artist, on='ArtistId', how='left')
    .merge(df_genre, on='GenreId', how='left')
    .merge(df_mediatype, on='MediaTypeId', how='left')
)

df_track.head(2)

,TrackId,Name_x,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice,Title,ArtistId,Name_y,Name,MediaTypeName
0,1,For Those About To Rock (We Salute You),1,1,1,"Angus Young, Malcolm Young, Brian Johnson",343719,11170334,0.99,For Those About To Rock We Salute You,1,AC/DC,Rock,MPEG audio file
1,2,Balls to the Wall,2,2,1,"U. Dirkschneider, W. Hoffmann, H. Frank, P. Ba...",342562,5510424,0.99,Balls to the Wall,2,Accept,Rock,Protected AAC audio file


In [72]:
# select and rename columns to keep only what we need
df_track = df_track[['TrackId', 'Name_x', 'Title', 'Name_y', 'Name', 'MediaTypeName', 'UnitPrice']]
df_track.columns = ['TrackId', 'TrackName', 'AlbumTitle', 'ArtistName', 'GenreName', 'MediaType', 'UnitPrice']
df_track.head(10)

,TrackId,TrackName,AlbumTitle,ArtistName,GenreName,MediaType,UnitPrice
0,1,For Those About To Rock (We Salute You),For Those About To Rock We Salute You,AC/DC,Rock,MPEG audio file,0.99
1,2,Balls to the Wall,Balls to the Wall,Accept,Rock,Protected AAC audio file,0.99
2,3,Fast As a Shark,Restless and Wild,Accept,Rock,Protected AAC audio file,0.99
3,4,Restless and Wild,Restless and Wild,Accept,Rock,Protected AAC audio file,0.99
4,5,Princess of the Dawn,Restless and Wild,Accept,Rock,Protected AAC audio file,0.99
5,6,Put The Finger On You,For Those About To Rock We Salute You,AC/DC,Rock,MPEG audio file,0.99
6,7,Let's Get It Up,For Those About To Rock We Salute You,AC/DC,Rock,MPEG audio file,0.99
7,8,Inject The Venom,For Those About To Rock We Salute You,AC/DC,Rock,MPEG audio file,0.99
8,9,Snowballed,For Those About To Rock We Salute You,AC/DC,Rock,MPEG audio file,0.99
9,10,Evil Walks,For Those About To Rock We Salute You,AC/DC,Rock,MPEG audio file,0.99


#### Load the Transformed DataFrames into the New Data Warehouse by Creating New Tables

In [73]:
df_customer = df_customer.reset_index().rename(columns={'index': 'customer_key'})
df_customer['customer_key'] += 1

df_track = df_track.reset_index().rename(columns={'index': 'track_key'})
df_track['track_key'] += 1

In [74]:
db_operation = "insert"

tables = [
    ('dim_customer', df_customer, 'customer_key'),
    ('dim_track', df_track, 'track_key'),
]

In [75]:
for table_name, dataframe, primary_key in tables:
    set_dataframe(uid, pwd, host_name, dst_dbname, dataframe, table_name, primary_key, db_operation)

#### Create Fact Table with Pandas Dataframes

First, extract invoice + invoiceline data.

In [76]:
sql_invoice = "SELECT * FROM chinook.invoice;"
df_invoice = get_dataframe(uid, pwd, host_name, src_dbname, sql_invoice)
df_invoice.head(2)

,InvoiceId,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total
0,1,2,2021-01-01,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98
1,2,4,2021-01-02,Ullevålsveien 14,Oslo,None,Norway,0171,3.96


In [77]:
sql_invoiceline = "SELECT * FROM chinook.invoiceline;"
df_invoiceline = get_dataframe(uid, pwd, host_name, src_dbname, sql_invoiceline)
df_invoiceline.head(2)

,InvoiceLineId,InvoiceId,TrackId,UnitPrice,Quantity
0,1,1,2,0.99,1
1,2,1,4,0.99,1


#### Join Invoice + Invoice Line DataFrames

In [78]:
df_fact_sales = df_invoiceline.merge(df_invoice, on='InvoiceId', how='left')
df_fact_sales.head(2)

,InvoiceLineId,InvoiceId,TrackId,UnitPrice,Quantity,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total
0,1,1,2,0.99,1,2,2021-01-01,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98
1,2,1,4,0.99,1,2,2021-01-01,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98


In [79]:
df_fact_sales.head(10)

,InvoiceLineId,InvoiceId,TrackId,UnitPrice,Quantity,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total
0,1,1,2,0.99,1,2,2021-01-01,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98
1,2,1,4,0.99,1,2,2021-01-01,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98
2,3,2,6,0.99,1,4,2021-01-02,Ullevålsveien 14,Oslo,None,Norway,0171,3.96
3,4,2,8,0.99,1,4,2021-01-02,Ullevålsveien 14,Oslo,None,Norway,0171,3.96
4,5,2,10,0.99,1,4,2021-01-02,Ullevålsveien 14,Oslo,None,Norway,0171,3.96
5,6,2,12,0.99,1,4,2021-01-02,Ullevålsveien 14,Oslo,None,Norway,0171,3.96
6,7,3,16,0.99,1,8,2021-01-03,Grétrystraat 63,Brussels,None,Belgium,1000,5.94
7,8,3,20,0.99,1,8,2021-01-03,Grétrystraat 63,Brussels,None,Belgium,1000,5.94
8,9,3,24,0.99,1,8,2021-01-03,Grétrystraat 63,Brussels,None,Belgium,1000,5.94
9,10,3,28,0.99,1,8,2021-01-03,Grétrystraat 63,Brussels,None,Belgium,1000,5.94


#### Lookup the Primary Keys from the Dimension Tables

##### Fetch the Primary Key and Business Key from the Date Dimension Table.

In [81]:
sql_dim_date = "SELECT date_key, full_date FROM chinook_dw.dim_date;"
df_dim_date = get_dataframe(uid, pwd, host_name, src_dbname, sql_dim_date)
df_dim_date.full_date = df_dim_date.full_date.astype('datetime64[ns]').dt.date
df_dim_date.head(2)

,date_key,full_date
0,20000101,2000-01-01
1,20000102,2000-01-02


In [82]:
# Lookup the Surrogate Primary Key (date_key) that Corresponds to the "InvoiceDate" Column.
df_dim_invoice_date = df_dim_date.rename(columns={"date_key": "invoice_date_key", "full_date": "InvoiceDate"})
df_fact_sales["InvoiceDate"] = df_fact_sales["InvoiceDate"].astype("datetime64[ns]").dt.date

df_fact_sales = pd.merge(df_fact_sales, df_dim_invoice_date, on="InvoiceDate", how="left")
df_fact_sales.drop(["InvoiceDate"], axis=1, inplace=True)
df_fact_sales.head(2)

,InvoiceLineId,InvoiceId,TrackId,UnitPrice,Quantity,CustomerId,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total,invoice_date_key
0,1,1,2,0.99,1,2,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98,20210101
1,2,1,4,0.99,1,2,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98,20210101


In [83]:
sql_dim_customer = "SELECT customer_key, CustomerId FROM chinook_dw.dim_customer;"
df_dim_customer = get_dataframe(uid, pwd, host_name, src_dbname, sql_dim_customer)
df_dim_customer.head(2)

,customer_key,CustomerId
0,1,1
1,2,2


In [84]:
sql_dim_track = "SELECT track_key, TrackId FROM chinook_dw.dim_track;"
df_dim_track = get_dataframe(uid, pwd, host_name, src_dbname, sql_dim_track)
df_dim_track.head(2)

,track_key,TrackId
0,1,1
1,2,2


In [85]:
# 1. Modify 'df_fact_sales' by merging it with 'df_dim_customer' on the 'CustomerId' column
# 2. Drop the 'CustomerId' column
# 3. Display the first 2 rows of the dataframe to validate your work

df_fact_sales = pd.merge(df_fact_sales, df_dim_customer, on='CustomerId', how='left')
df_fact_sales.drop(['CustomerId'], axis=1, inplace=True)
df_fact_sales.head(2)

,InvoiceLineId,InvoiceId,TrackId,UnitPrice,Quantity,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total,invoice_date_key,customer_key
0,1,1,2,0.99,1,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98,20210101,2
1,2,1,4,0.99,1,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98,20210101,2


In [86]:
# do the same w/ df_dim_track

df_fact_sales = pd.merge(df_fact_sales, df_dim_track, on='TrackId', how='left')
df_fact_sales.drop(['TrackId'], axis=1, inplace=True)
df_fact_sales.head(2)

,InvoiceLineId,InvoiceId,UnitPrice,Quantity,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total,invoice_date_key,customer_key,track_key
0,1,1,0.99,1,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98,20210101,2,2
1,2,1,0.99,1,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98,20210101,2,4


#### Perform Necessary Transformations and Create Fact Sale Key

In [87]:
ordered_columns = ['InvoiceLineId', 'InvoiceId', 'customer_key', 'track_key', 'invoice_date_key', 'UnitPrice', 'Quantity', 'BillingAddress',
                  'BillingCity', 'BillingState', 'BillingCountry', 'BillingPostalCode', 'Total']
df_fact_sales = df_fact_sales[ordered_columns]

df_fact_sales.insert(0, "fact_sale_key", range(1, df_fact_sales.shape[0]+1))
df_fact_sales.head(2)

,fact_sale_key,InvoiceLineId,InvoiceId,customer_key,track_key,invoice_date_key,UnitPrice,Quantity,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total
0,1,1,1,2,2,20210101,0.99,1,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98
1,2,2,1,2,4,20210101,0.99,1,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98


#### Load Newly Transformed Data into the Chinook_DW Data Warehouse

In [88]:
table_name = "fact_sales"
primary_key = "fact_sale_key"
db_operation = "insert"

set_dataframe(uid, pwd, host_name, dst_dbname, df_fact_sales, table_name, primary_key, db_operation)

### SQL QUERIES

#### Query 1: Total sales per customer

In [89]:
sql_test = """
SELECT 
    c.FirstName AS first_name,
    c.LastName AS last_name,
    SUM(f.Quantity) AS total_quantity,
    SUM(f.Quantity * f.UnitPrice) AS total_sales
FROM {0}.fact_sales f
JOIN {0}.dim_customer c
    ON f.customer_key = c.customer_key
GROUP BY c.FirstName, c.LastName
ORDER BY total_sales DESC;
""".format(dst_dbname)

df_test = get_dataframe(uid, pwd, host_name, dst_dbname, sql_test)
df_test.head()

,first_name,last_name,total_quantity,total_sales
0,Helena,Holý,38.0,49.62
1,Richard,Cunningham,38.0,47.62
2,Luis,Rojas,38.0,46.62
3,Hugh,O'Reilly,38.0,45.62
4,Ladislav,Kovács,38.0,45.62


#### Query 2: Total sales per track (top songs)

In [90]:
sql_tracks = """
SELECT 
    t.TrackName AS track_name,
    SUM(f.Quantity) AS total_quantity_sold,
    SUM(f.Quantity * f.UnitPrice) AS total_revenue
FROM {0}.fact_sales f
JOIN {0}.dim_track t
    ON f.track_key = t.track_key
GROUP BY t.TrackName
ORDER BY total_revenue DESC
LIMIT 10;
""".format(dst_dbname)

df_tracks = get_dataframe(uid, pwd, host_name, dst_dbname, sql_tracks)
df_tracks.head()


,track_name,total_quantity_sold,total_revenue
0,The Trooper,5.0,4.95
1,Dazed and Confused,5.0,4.95
2,Pilot,2.0,3.98
3,Walkabout,2.0,3.98
4,How to Stop an Exploding Man,2.0,3.98


#### Query 3: Sales over time (by month)

In [91]:
sql_time = """
SELECT 
    d.calendar_year,
    d.month_name,
    COUNT(f.Quantity) AS monthly_sales,
    SUM(f.Quantity * f.UnitPrice) AS monthly_revenue
FROM {0}.fact_sales f
JOIN {0}.dim_date d
    ON f.invoice_date_key = d.date_key
GROUP BY d.calendar_year, d.month_name
ORDER BY d.calendar_year, MIN(d.full_date);
""".format(dst_dbname)

df_time = get_dataframe(uid, pwd, host_name, dst_dbname, sql_time)
df_time.head()


,calendar_year,month_name,monthly_sales,monthly_revenue
0,2021,January,36,35.64
1,2021,February,38,37.62
2,2021,March,38,37.62
3,2021,April,38,37.62
4,2021,May,38,37.62


#### TODO:
- make names of columns more consistent (make everything camelCase since most columns are like that already)
- add some more headers throughout ..
- readme
- set up github repo